# SocialEngineerArena Training + Hub Push

This notebook performs **real model training** and pushes the trained checkpoint to Hugging Face Hub.

It uses the project training entrypoint (`scripts/train_suggest_model.py`), which runs SFT on the environment-aligned JSON action format and writes training artifacts (`loss_curve.png`, `summary.json`) plus model weights to your Hub repo.

In [1]:
# If running from notebooks/, move to repo root first.
import os
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

!pip install -q -U pip
!pip install -q huggingface_hub openenv-core trl transformers accelerate datasets matplotlib
!pip install -q -e .

print("Repo root:", Path.cwd())

D:\Openenv-final\social-engineer-arena
Repo root: D:\Openenv-final\social-engineer-arena


In [2]:
import json
import os
from pathlib import Path

HF_TOKEN = os.getenv("HF_TOKEN", "").strip()
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN before running training (PowerShell: $env:HF_TOKEN='hf_xxx').")

# ---- Training configuration ----
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.setdefault("MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
os.environ.setdefault("OUTPUT_REPO", "vinod2005/social-engineer-arena-suggest")
os.environ.setdefault("OUTPUT_DIR", "outputs/social_engineer_arena_suggest")
os.environ.setdefault("SCENARIOS_PATH", "social_engineer_arena/data/scenarios.large.json")
os.environ.setdefault("MAX_STEPS", "120")
os.environ.setdefault("DATA_MULTIPLIER", "12")
os.environ.setdefault("EVAL_STRATEGY", "no")
os.environ.setdefault("MAX_EVAL_SCENARIOS", "80")
os.environ.setdefault("PUSH_TO_HUB", "1")

print("MODEL_NAME:", os.environ["MODEL_NAME"])
print("OUTPUT_REPO:", os.environ["OUTPUT_REPO"])
print("SCENARIOS_PATH:", os.environ["SCENARIOS_PATH"])
print("MAX_STEPS:", os.environ["MAX_STEPS"])
print("DATA_MULTIPLIER:", os.environ["DATA_MULTIPLIER"])

scenarios_path = Path(os.environ["SCENARIOS_PATH"])
if not scenarios_path.exists():
    raise FileNotFoundError(f"Dataset not found: {scenarios_path}")
print("Dataset file size (MB):", round(scenarios_path.stat().st_size / (1024 * 1024), 2))

MODEL_NAME: Qwen/Qwen2.5-0.5B-Instruct
OUTPUT_REPO: vinod2005/social-engineer-arena-suggest
SCENARIOS_PATH: social_engineer_arena/data/scenarios.large.json
MAX_STEPS: 120
DATA_MULTIPLIER: 12
Dataset file size (MB): 7.8


## Submit training to Hugging Face Jobs (cloud GPU)

The next cell launches a **remote HF Job** (no local training). The job will:

- build dataset from `SCENARIOS_PATH`
- fine-tune model on HF GPU
- push model + tokenizer to `OUTPUT_REPO`

After submission, monitor with `hf jobs ps` and `hf jobs logs <job_id>`.

In [ ]:
import shlex
import subprocess

# Pick hardware: t4-medium (budget) or a10g-large (faster / safer for larger runs)
HF_FLAVOR = os.getenv("HF_FLAVOR", "a10g-large")
HF_TIMEOUT = os.getenv("HF_TIMEOUT", "3h")

cmd = [
    "hf", "jobs", "uv", "run",
    "--flavor", HF_FLAVOR,
    "--timeout", HF_TIMEOUT,
    "--secrets", "HF_TOKEN",
    "--env", f"MODEL_NAME={os.environ['MODEL_NAME']}",
    "--env", f"OUTPUT_REPO={os.environ['OUTPUT_REPO']}",
    "--env", f"OUTPUT_DIR={os.environ['OUTPUT_DIR']}",
    "--env", f"SCENARIOS_PATH={os.environ['SCENARIOS_PATH']}",
    "--env", f"MAX_STEPS={os.environ['MAX_STEPS']}",
    "--env", f"DATA_MULTIPLIER={os.environ['DATA_MULTIPLIER']}",
    "--env", f"EVAL_STRATEGY={os.environ['EVAL_STRATEGY']}",
    "--env", f"MAX_EVAL_SCENARIOS={os.environ['MAX_EVAL_SCENARIOS']}",
    "--env", f"PUSH_TO_HUB={os.environ['PUSH_TO_HUB']}",
    "scripts/train_suggest_model.py",
]

print("Submitting job:")
print(" ".join(shlex.quote(x) for x in cmd))
res = subprocess.run(cmd, text=True)
if res.returncode != 0:
    raise RuntimeError(f"HF Job submission failed with exit code {res.returncode}")

print("Submitted. Use `hf jobs ps` to get job id, then `hf jobs logs <job_id>`.")

Running: C:\Users\vinod\anaconda3\python.exe scripts/train_suggest_model.py


In [3]:
# Optional: smoke-test the pushed model (Hub inference API)
import subprocess
import sys

smoke_cmd = [
    sys.executable,
    "scripts/run_suggest_action.py",
    "--model-id",
    os.environ["OUTPUT_REPO"],
    "--episodes",
    "3",
]
print("Running:", " ".join(smoke_cmd))
_ = subprocess.run(smoke_cmd, text=True)

Endpoint verification complete if rewards are non-trivial and vary across scenarios.
Next step: plug this reward pipeline into GRPOTrainer on GPU compute.


## Push the environment to Hugging Face Space

Run the next cell from repo root after training if you want to publish the full environment demo.

Make sure your target Space already exists (for example `vinod2005/social-engineer-arena`).

In [ ]:
# Requires: `pip install -U openenv`
# Uses HF_TOKEN from environment.

SPACE_ID = os.getenv("SPACE_ID", "vinod2005/social-engineer-arena")
print("Pushing environment to:", SPACE_ID)

!openenv push --repo-id {SPACE_ID} --yes